# Wave-following average-- Winds, precipitation, and moisture
This notebook produces wave-following averages of several fields, using the wavetracks from QTRACK. Select the wave in question (whether Paulette 'second', or Rene 'main'). 
Plots for a selected ensemble set, whether fluxon, fluxoff, rst_on24, 
rst_on36, or rst_on48. 

In [2]:
import pkg_resources
from __future__ import print_function
from datetime import datetime, timedelta

import numpy as np
import xarray as xr
import pandas as pd
import os
import glob
from netCDF4 import Dataset, num2date, date2num

#from AEW_module import season, AEW, AEW_CCKW

from metpy.units import units
import geocat.viz as gv


import metpy

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import matplotlib.ticker as mticker 

import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter
import wrf
from wrf import (to_np, interplevel, geo_bounds, getvar, smooth2d, get_cartopy, cartopy_xlim,
                 cartopy_ylim, latlon_coords, destagger)

import seaborn as sns
import metpy.calc as mpcalc

In [3]:
plotsdir = '/glade/u/home/athornton/qtrack/'

In [4]:
## The following are lists to select each ensemble set and respective initialization time
variation = ['fluxon', 'rst_on24', 'rst_on36', 'rst_on48', '', 'fluxoff', '', '', '']
init_times = ['0300','0303','0306',
             '0309','0312','0315',
             '0318','0321','0400']

In [5]:
#########################################
### MAKE SELECTIONS HERE
#set = 'Fluxes on'          # try: '24 hours without fluxes'
set_name = variation[1]     # try: variation[1]
wave = 'second'             # 'second' == Paulette or 'main' == Rene
title = '24 hrs decoupled'  # What will be displayed on main plot

In [6]:
## This is for our column moisture calculation
def mass_weighted_vert_integral(height, data):
    # data is expected to be on pressure levels
    levels = p.bottom_top
    deltaP = (levels - levels.shift(bottom_top=1))
    vert_int_data = ((data.shift(bottom_top=1)+data)*.5*deltaP).sum(dim='bottom_top') / metpy.constants.earth_gravity    
    return  vert_int_data

In [ ]:
# Set up loop to run same plots for multiple different times (assume here that we have 1 time per wrfout file)
plt.rcParams['savefig.dpi'] = 255

fig, ax = plt.subplots(figsize=(15,5))

ax.set_title('Wave-following Area Average\n Column Moisture, Precipitation, & Surface Windspeed', size=15, weight='bold')
ax.set_ylabel('$1/s$', size =13)
ax.set_xlabel('Date', size=13)
date_form = DateFormatter('%b-%d')
ax.xaxis.set_major_formatter(date_form)
# column moisture
ax.set_ylim(0.02,0.035)

# surface windspeed
ax2=ax.twinx()
ax2.set_ylim(-2,40)

# precipitation
ax3=ax.twinx()
ax3.set_ylim(-0.2,5)


winds_list_tot = []
moist_list_tot = []
rain_list_tot = []

for init_time in init_times:
    save_name = set_name +"_"+ init_time
    print(save_name)
    # get wavetrack data
    df = xr.open_dataset('/glade/u/home/athornton/qtrack/'+wave+'_wave/'+wave+'_wave_track_'+save_name+'.nc')

    # get wrfout data
    os.chdir("/glade/campaign/univ/uncs0067/flux_experiments/cent_atl_case/init"+init_time+"z/"+set_name+"/")
    datafiles = sorted(glob.glob("./wrfout*"))
    numfiles=len(datafiles)
    # select every other file to match wave-track resolution
    datafiles_6hr = datafiles[::2]
    print(numfiles)
    print(datafiles[0])
    start_time ='2020-09-'+init_time[:2]+' '+init_time[2:]+':00'
    times_list = pd.date_range(start=start_time, end='2020-09-09 12:00', freq='6h')
    # select lons, lats in the window of our simulation time
    wave_track_lons = df.lon[-len(times_list):]
    wave_track_lats = df.lat[-len(times_list):]
    wrf_out_data = xr.open_dataset(datafiles_6hr[0])  
    p =wrf_out_data['P']

    winds_list = []
    moist_list = []
    for i in range(0,len(datafiles_6hr)):
        ncfile = Dataset(datafiles_6hr[i])
        Time=wrf.extract_times(ncfile, timeidx=0, method='cat', squeeze=True, cache=None, meta=False, do_xtime=False)
        timestr=(str(Time))
        # Set up one time string for plot titles, another for file names
        titletime=(timestr[0:10]+' '+timestr[11:16])
        filetime=(timestr[0:10]+'_'+timestr[11:16])
        print('WRF valid time: ',filetime)
    
        wrf_out_data = xr.open_dataset(datafiles_6hr[i])  
        
        # Get all the variables we need
        qwv = wrf_out_data["QVAPOR"]
        ua = wrf_out_data["U"]
        va = wrf_out_data["V"]
    
        lats, lons = latlon_coords(wrf_out_data['P'])
        lats_np = to_np(lats)
        lons_np = to_np(lons)
        lats_np = lats_np[0,:,0]
        lons_np = lons_np[0,0,:]
        
        # Unstagger the winds to match the pressure grid
        ua = destagger(ua, stagger_dim=3)             # Unstagger U in the x-direction
        va = destagger(va, stagger_dim=2)             # Unstagger V in the x-direction
    
        # Grid spacing for vorticity calculation
        dx = ncfile.DX * units("m")
        dy = ncfile.DY * units("m")
        
        u_winds = xr.DataArray(ua)[0]
        v_winds = xr.DataArray(va)[0]
        qwv =  xr.DataArray(qwv)[0]
        
        u_sfc = u_winds.sel(dim_1=1)*units('m/s')     # select surface winds
        v_sfc = v_winds.sel(dim_1=1)*units('m/s')     # select surface winds
        wind_sfc = mpcalc.wind_speed(u_sfc, v_sfc)

        # Lat, Lon cooridante of center of square we are interested in
        lat_lon = [wave_track_lats[i],wave_track_lons[i]]                
        
        x_ycenter = wrf.ll_to_xy(ncfile, lat_lon[0], lat_lon[1])
    
        # specify how big you want the wave-following box to be
        delta = 50                             # in grid points 
    
        winds = wind_sfc[int(x_ycenter[1]-delta):int(x_ycenter[1]+delta),int(x_ycenter[0]-delta):int(x_ycenter[0]+delta)]
    
        # wave tracks have nans at the beginning so return nan if it's the beginning of the track
        if not winds.any():
            max_wind = np.nan
        else:
            max_wind = np.max(winds.values)
    
        columnq = mass_weighted_vert_integral(p, qwv)
        avg_qwv = columnq[int(x_ycenter[1]-delta):int(x_ycenter[1]+delta),int(x_ycenter[0]-delta):int(x_ycenter[0]+delta)].mean()
    
        winds_list.append(max_wind)
        moist_list.append(avg_qwv.values)
        
    # Column moisture
    date_list = pd.date_range(start=start_time, end='2020-09-09 12:00:00', freq='6h')
    df=pd.DataFrame(moist_list)
    df['date']=date_list
    df=df.set_index('date')
    moist_list_tot.append(df)

    # Surface windspeed
    df2=pd.DataFrame(winds_list)
    df2['date']=date_list
    df2=df2.set_index('date')
    winds_list_tot.append(df2)
    
    # Precipitation
    data = xr.open_mfdataset(datafiles_6hr, concat_dim='Time', combine='nested', engine='netcdf4')
    rain = data.RAINC
    time = data.XTIME.compute()
    rain_list = [np.zeros_like(rain[0])]  # Start with a zero array for the first time step

    for i in range(1, len(rain)):
        rain_hour = rain[i] - rain[i-1]
        rain_hour = np.maximum(rain_hour, 0)  # Set negative values to 0
        rain_list.append(rain_hour)

    rain = xr.DataArray(rain_list)

    rain_box_list = []
    for i in range(0,len(datafiles_6hr)):
        lat_lon = [wave_track_lats[i],wave_track_lons[i]]      # Lat, Lon cooridante of center of square we are interested in
        x_ycenter = wrf.ll_to_xy(ncfile, lat_lon[0], lat_lon[1])

        # specify how big you want the wave-following box to be
        delta = 50                                   # in grid points 
        rain_inc = rain[i]
        rain_box = rain_inc[int(x_ycenter[1]-delta):int(x_ycenter[1]+delta),int(x_ycenter[0]-delta):int(x_ycenter[0]+delta)].mean()
        rain_box_list.append(rain_box.values)

    df3=pd.DataFrame(rain_box_list)
    df3['date']=date_list
    df3=df3.set_index('date')
    rain_list_tot.append(df3)

    ax.plot(date_list, moist_list, color='forestgreen', alpha=0.5)
    ax2.plot(date_list, winds_list, color='darkgray', alpha=0.5)
    ax3.plot(date_list, rain_box_list, color='lightsteelblue', alpha=0.5)

ax.set_ylabel('Column Moisture', size=13, color='seagreen',alpha=0.8)
    
ax2.set_ylabel('Windspeed', size=13, color='black')
ax2.legend(loc='lower right')

ax3.spines['right'].set_position(('outward', 60))
ax3.set_ylabel('Precip (mm)', size=13, color='steelblue')
    
ax2.axhline(y=17.5, color='lightgray')

ens_avg = pd.concat(moist_list_tot, join='outer', axis=1).fillna(np.nan)
ens_avg = ens_avg.mean(axis=1)
ax.plot(ens_avg, color='forestgreen', label='Column q ens average', linestyle=':', linewidth=2.)
ax.legend(loc='lower left')

ens_avg2 = pd.concat(winds_list_tot, join='outer', axis=1).fillna(np.nan)
ens_avg2 = ens_avg2.mean(axis=1)
ax2.plot(ens_avg2, color='black', label='Sfc winds ens average', linewidth=2.)
ax2.legend(loc='lower right')

ens_avg3 = pd.concat(rain_list_tot, join='outer', axis=1).fillna(np.nan)
ens_avg3 = ens_avg3.mean(axis=1)
ax3.plot(ens_avg3, color='steelblue', label='Precip ens average', linestyle='-.', linewidth=2.)
ax3.legend(loc='upper left')

ax.grid(color='linen')
#ax.legend(loc='lower right')
#ax.set_ylim(-0.000005,0.00005)
#plt.savefig(plotsdir+'wave_following_ens_moist_wind_'+set_name+'_'+wave+'.png')


rst_on24_0300
53
./wrfout_d01_2020-09-03_00:00:00
WRF valid time:  2020-09-03_00:00
WRF valid time:  2020-09-03_06:00
WRF valid time:  2020-09-03_12:00
WRF valid time:  2020-09-03_18:00
WRF valid time:  2020-09-04_00:00
WRF valid time:  2020-09-04_06:00
WRF valid time:  2020-09-04_12:00
WRF valid time:  2020-09-04_18:00
WRF valid time:  2020-09-05_00:00
WRF valid time:  2020-09-05_06:00
WRF valid time:  2020-09-05_12:00
WRF valid time:  2020-09-05_18:00
WRF valid time:  2020-09-06_00:00
WRF valid time:  2020-09-06_06:00
WRF valid time:  2020-09-06_12:00
WRF valid time:  2020-09-06_18:00
WRF valid time:  2020-09-07_00:00
WRF valid time:  2020-09-07_06:00
WRF valid time:  2020-09-07_12:00
WRF valid time:  2020-09-07_18:00
WRF valid time:  2020-09-08_00:00
